## 1. Configuration

In [ ]:
import os, re, json, gzip, time, hashlib, random, tomllib, getpass, urllib.parse
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone, timedelta
from pathlib import Path

import requests
import pandas as pd
from tqdm.auto import tqdm

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")  # set this in your environment, never in code

# ---- GH Archive: one hour (15:00 UTC) every 6 days; sparse/missing hours are discarded
GH_START, GH_END, GH_STEP_DAYS, GH_HOUR_UTC = "2026-09-01", "2026-11-12", 6, 15
MIN_EVENTS_PER_HOUR = 20_000

# ---- Scale
N_CANDIDATES = 2500          # accounts tested by the prefilter (≈ 1 quota point per 10 accounts)
MIN_MANIFEST_COMMITS = 20    # cumulative commits on package.json / requirements.txt / pyproject.toml to qualify
N_DEVS_FINAL = 800           # developers kept for the full extraction (set to 50 for a quick test)
SEED = 42

# ---- Extraction parameters
REPOS_PER_DEV = 20           # personal non-fork repos, most recently pushed
HIST_PER_FILE = 100          # max commits read per manifest
STARS_PAGES = 2              # 2 x 100 most recent stars per developer
MAX_EVENTS_PER_USER_PER_HOUR = 200
BULK_THRESHOLD = 10          # a commit that adds more than N packages at once is flagged is_bulk

DATA_DIR, CACHE_DIR, OUT_DIR = Path("../data/raw/gharchive"), Path("../data/raw/cache"), Path("../data/interim")
META_DIR = CACHE_DIR / "meta"
for d in (DATA_DIR, CACHE_DIR, OUT_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

def hours_list():
    d, end, out = datetime.fromisoformat(GH_START), datetime.fromisoformat(GH_END), []
    while d <= end:
        out.append(f"{d:%Y-%m-%d}-{GH_HOUR_UTC}")
        d += timedelta(days=GH_STEP_DAYS)
    return out
GH_HOURS = hours_list()
print(len(GH_HOURS), "candidate hours |", N_CANDIDATES, "prefiltered accounts |", N_DEVS_FINAL, "target developers")

13 heures candidates | 2500 comptes préfiltrés | 800 développeurs visés


## 2. Rate limit check

In [4]:
r = requests.get("https://api.github.com/rate_limit", headers={"Authorization": f"Bearer {GITHUB_TOKEN}"})
print(r.status_code, r.json()["resources"]["graphql"])

200 {'limit': 5000, 'used': 0, 'remaining': 5000, 'reset': 1795019836}


## 3. Candidate accounts from GH Archive

In [ ]:
def download_hour(h):
    path = DATA_DIR / f"{h}.json.gz"
    if path.exists() and path.stat().st_size > 0:
        return path
    url = f"https://data.gharchive.org/{h}.json.gz"
    tmp = path.with_suffix(".part")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            tmp.rename(path)
            return path
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                return None
            time.sleep(2 ** attempt)
        except Exception:
            time.sleep(2 ** attempt)
    return None

CANDIDATE_TYPES = {"PushEvent", "PullRequestEvent", "IssuesEvent", "CreateEvent", "PullRequestReviewEvent"}
is_bot = lambda l: l.lower().endswith(("[bot]", "-bot", "_bot")) or "bot" in l.lower().split("-")

per_hour_user, push_users, dense_hours = Counter(), Counter(), []

for h in tqdm(GH_HOURS, desc="GH Archive hours"):
    p = download_hour(h)
    if p is None:
        print(f"  {h}: missing from archive")
        continue
    n_lines, hour_counts, hour_push = 0, Counter(), Counter()
    try:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                n_lines += 1
                try:
                    ev = json.loads(line)
                except ValueError:
                    continue
                t, login = ev.get("type"), (ev.get("actor") or {}).get("login")
                if not login or t not in CANDIDATE_TYPES:
                    continue
                hour_counts[login] += 1
                if t == "PushEvent":
                    hour_push[login] += 1
    except (EOFError, OSError):
        print(f"  {h}: corrupted file -> deleted, re-run the cell")
        p.unlink(missing_ok=True)
        continue
    if n_lines < MIN_EVENTS_PER_HOUR:
        print(f"  {h}: {n_lines:,} events -> sparse hour, skipped")
        continue
    dense_hours.append(h)
    for l, c in hour_counts.items():
        per_hour_user[(h, l)] = c
    push_users.update(hour_push)
    print(f"  {h}: {n_lines:,} events kept")

assert dense_hours, "No dense hour: lower MIN_EVENTS_PER_HOUR or change the dates"
too_active = {l for (h, l), c in per_hour_user.items() if c > MAX_EVENTS_PER_USER_PER_HOUR}
candidates = sorted(l for l in push_users if not is_bot(l) and l not in too_active)

rng = random.Random(SEED)
cand_sample = rng.sample(candidates, min(N_CANDIDATES, len(candidates)))
print(f"\n{len(dense_hours)} dense hours | {len(candidates):,} eligible accounts | {len(cand_sample):,} sampled for the prefilter")

Heures GH Archive:   0%|          | 0/13 [00:00<?, ?it/s]

  2026-09-01-15: 1,844 événements -> heure creuse, ignorée
  2026-09-07-15: 73,767 événements retenus
  2026-09-13-15: 128,527 événements retenus
  2026-09-19-15: 120,341 événements retenus
  2026-09-25-15: 72,191 événements retenus
  2026-10-01-15: 85,569 événements retenus
  2026-10-07-15: 6,219 événements -> heure creuse, ignorée
  2026-10-13-15: 80,662 événements retenus
  2026-10-19-15: 45,153 événements retenus
  2026-10-25-15: 85,835 événements retenus
  2026-10-31-15: 4,368 événements -> heure creuse, ignorée
  2026-11-06-15: 3,146 événements -> heure creuse, ignorée
  2026-11-12-15: 3,785 événements -> heure creuse, ignorée

8 heures denses | 189,120 comptes éligibles | 2,500 tirés pour le préfiltre


## 4. GraphQL client

In [ ]:
GQL_URL = "https://api.github.com/graphql"
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}"})
api_stats = {"calls": 0, "cached": 0, "cost": 0, "remaining": None}

def _sleep_until(reset_iso, extra=5):
    reset = datetime.fromisoformat(reset_iso.replace("Z", "+00:00"))
    wait = max(0, (reset - datetime.now(timezone.utc)).total_seconds()) + extra
    print(f"  GraphQL quota low: sleeping {wait:.0f}s")
    time.sleep(wait)

def gql(query, variables=None):
    variables = variables or {}
    key = hashlib.sha1(json.dumps([query, variables], sort_keys=True).encode()).hexdigest()
    cpath = CACHE_DIR / f"{key}.json"
    if cpath.exists():
        api_stats["cached"] += 1
        return json.loads(cpath.read_text())

    last = None
    for attempt in range(6):
        try:
            r = session.post(GQL_URL, json={"query": query, "variables": variables}, timeout=90)
        except requests.RequestException as e:
            last = repr(e)[:200]
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 401:
            raise RuntimeError("Invalid token (401)")
        if r.status_code in (502, 503, 504):
            last = f"HTTP {r.status_code} (request too heavy / GitHub timeout)"
            time.sleep(2 ** attempt)
            continue
        if r.status_code in (403, 429):
            last = f"HTTP {r.status_code} (rate limit)"
            ra = r.headers.get("Retry-After")
            reset = r.headers.get("X-RateLimit-Reset")
            if ra:
                time.sleep(int(ra) + 1)
            elif reset and r.headers.get("X-RateLimit-Remaining") == "0":
                time.sleep(max(0, int(reset) - time.time()) + 5)
            else:
                time.sleep(60 * (attempt + 1))   # secondary limit
            continue
        if r.status_code != 200:
            last = f"HTTP {r.status_code}: {r.text[:150]}"
            time.sleep(2 ** attempt)
            continue

        payload = r.json()
        errs = payload.get("errors") or []
        if any(e.get("type") == "RATE_LIMITED" for e in errs):
            time.sleep(60)
            continue
        if payload.get("data") is None and errs:
            last = str(errs)[:300]
            time.sleep(2 ** attempt)
            continue

        api_stats["calls"] += 1
        rl = (payload.get("data") or {}).get("rateLimit")
        if rl:
            api_stats["cost"] += rl["cost"]
            api_stats["remaining"] = rl["remaining"]
            if rl["remaining"] < 100:
                _sleep_until(rl["resetAt"])
        cpath.write_text(json.dumps(payload))
        return payload
    raise RuntimeError(f"GraphQL: failed after 6 attempts ({last})")

me = gql("query { viewer { login } rateLimit { cost remaining resetAt } }")
print("Logged in as:", me["data"]["viewer"]["login"], "| remaining quota:", api_stats["remaining"])

Connecté en tant que : BoutalebAnas | quota restant : None


## 5. Prefilter: who has enough manifest history?

A request for 10 accounts (= 1 quota point): for each, it retrieves their personal, non-forked repositories and the **number of commits** affecting each manifest file. No content is read here.

In [ ]:
def prefilter_query(logins):
    parts = []
    for i, l in enumerate(logins):
        parts.append(f"""u{i}: user(login: {json.dumps(l)}) {{
          login createdAt followers {{ totalCount }}
          repositories(first: 30, ownerAffiliations: [OWNER], isFork: false, privacy: PUBLIC,
                       orderBy: {{field: PUSHED_AT, direction: DESC}}) {{
            nodes {{ defaultBranchRef {{ target {{ ... on Commit {{
              hp: history(path: "package.json", first: 1) {{ totalCount }}
              hr: history(path: "requirements.txt", first: 1) {{ totalCount }}
              hy: history(path: "pyproject.toml", first: 1) {{ totalCount }}
            }} }} }} }}
          }} }}""")
    return "query { rateLimit { cost remaining resetAt } " + " ".join(parts) + " }"

pre_rows, pre_failed = [], []
BATCH = 4                      # small batches: heavy queries exceed GitHub's timeout

def fetch_users(logins):
    """Return {login: user node}. If the batch fails, split it until the failing account is isolated."""
    try:
        data = gql(prefilter_query(logins)).get("data") or {}
        return {l: data.get(f"u{j}") for j, l in enumerate(logins)}
    except Exception as e:
        if len(logins) == 1:
            pre_failed.append((logins[0], str(e)[:200]))
            return {}
        mid = len(logins) // 2
        return {**fetch_users(logins[:mid]), **fetch_users(logins[mid:])}

for i in tqdm(range(0, len(cand_sample), BATCH), desc="Prefilter"):
    users = fetch_users(cand_sample[i:i + BATCH])
    for login, u in users.items():
        if not u:
            continue
        n_commits = n_repos = n_py = n_js = 0
        for r in u["repositories"]["nodes"]:
            t = ((r.get("defaultBranchRef") or {}).get("target")) or {}
            hp, hr, hy = [(t.get(k) or {}).get("totalCount", 0) for k in ("hp", "hr", "hy")]
            tot = hp + hr + hy
            n_commits += tot
            n_repos += tot > 0
            n_js += hp
            n_py += hr + hy
        pre_rows.append({"developer": login, "followers": u["followers"]["totalCount"],
                         "account_created_at": u["createdAt"], "n_repos_with_manifest": n_repos,
                         "n_manifest_commits": n_commits, "n_npm_commits": n_js, "n_pypi_commits": n_py})

if pre_failed:
    print(f"{len(pre_failed)} accounts skipped after isolated failure. Example error: {pre_failed[0]}")
pre = pd.DataFrame(pre_rows)
pre.to_csv(OUT_DIR / "prefilter.csv", index=False)
passing = pre[pre.n_manifest_commits >= MIN_MANIFEST_COMMITS]
print(f"{len(pre):,} accounts resolved | {len(passing):,} pass the threshold ({len(passing)/max(1,len(pre)):.0%}) | API: {api_stats}")

sample = passing.sample(min(N_DEVS_FINAL, len(passing)), random_state=SEED)["developer"].tolist()
print(f"Developers kept for the full extraction: {len(sample):,}")
if len(sample) < N_DEVS_FINAL:
    print("  -> fewer than target: raise N_CANDIDATES or lower MIN_MANIFEST_COMMITS")

Préfiltre:   0%|          | 0/625 [00:00<?, ?it/s]

2,320 comptes résolus | 580 passent le seuil (25%) | API : {'calls': 628, 'cached': 1, 'cost': 2500, 'remaining': 4058}
Développeurs retenus pour l'extraction complète : 580
  -> moins que visé : augmente N_CANDIDATES ou baisse MIN_MANIFEST_COMMITS


## 6. Manifest parsers

In [ ]:
KIND_PRIORITY = {"runtime": 0, "peer": 1, "optional": 2, "dev": 3}
PEP508 = re.compile(r"^\s*([A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?)")

def norm_py(name):
    return re.sub(r"[-_.]+", "-", name).lower()

def _py_name(spec):
    m = PEP508.match(spec)
    return norm_py(m.group(1)) if m else None

def _merge(out, name, kind):
    if name and (name not in out or KIND_PRIORITY[kind] < KIND_PRIORITY[out[name]]):
        out[name] = kind

def parse_requirements(txt):
    out = {}
    for line in txt.splitlines():
        line = line.split(" #")[0].strip()
        if not line or line.startswith(("#", "-", "git+", "http", ".", "/")):
            continue
        _merge(out, _py_name(line.split(";")[0]), "runtime")
    return out

def parse_pyproject(txt):
    out = {}
    try:
        d = tomllib.loads(txt)
    except Exception:
        return out
    proj = d.get("project", {}) or {}
    for s in proj.get("dependencies", []) or []:
        _merge(out, _py_name(s), "runtime")
    for grp in (proj.get("optional-dependencies", {}) or {}).values():
        for s in grp:
            _merge(out, _py_name(s), "optional")
    for grp in (d.get("dependency-groups", {}) or {}).values():
        for s in grp:
            if isinstance(s, str):
                _merge(out, _py_name(s), "dev")
    poetry = (d.get("tool", {}) or {}).get("poetry", {}) or {}
    for n in poetry.get("dependencies", {}) or {}:
        if n.lower() != "python":
            _merge(out, norm_py(n), "runtime")
    for n in poetry.get("dev-dependencies", {}) or {}:
        _merge(out, norm_py(n), "dev")
    for g in (poetry.get("group", {}) or {}).values():
        for n in (g.get("dependencies", {}) or {}):
            _merge(out, norm_py(n), "dev")
    return out

def parse_package_json(txt):
    out = {}
    try:
        d = json.loads(txt)
    except Exception:
        return out
    if not isinstance(d, dict):
        return out
    for field, kind in (("dependencies", "runtime"), ("peerDependencies", "peer"),
                        ("optionalDependencies", "optional"), ("devDependencies", "dev")):
        for n in (d.get(field) or {}):
            _merge(out, n.lower(), kind)
    return out

PARSERS = {
    "package.json":     ("npm",  parse_package_json),
    "requirements.txt": ("pypi", parse_requirements),
    "pyproject.toml":   ("pypi", parse_pyproject),
}
HIST_ALIAS = {"hp": "package.json", "hr": "requirements.txt", "hy": "pyproject.toml"}

## 7. Dated manifest history (retained developers)

In [9]:
REPOS_Q = """
fragment H on CommitHistoryConnection {
  totalCount
  pageInfo { hasNextPage }
  nodes { oid committedDate author { user { login } } }
}
query($login: String!, $n: Int!, $h: Int!) {
  rateLimit { cost remaining resetAt }
  user(login: $login) {
    login
    createdAt
    followers { totalCount }
    repositories(first: $n, ownerAffiliations: [OWNER], isFork: false, privacy: PUBLIC,
                 orderBy: {field: PUSHED_AT, direction: DESC}) {
      totalCount
      nodes {
        nameWithOwner isArchived createdAt pushedAt stargazerCount
        primaryLanguage { name }
        defaultBranchRef { target { ... on Commit {
          hp: history(path: "package.json", first: $h) { ...H }
          hr: history(path: "requirements.txt", first: $h) { ...H }
          hy: history(path: "pyproject.toml", first: $h) { ...H }
        } } }
      }
    }
  }
}
"""

def fetch_texts(owner, name, path, oids, chunk=30):
    texts = {}
    for i in range(0, len(oids), chunk):
        part = oids[i:i + chunk]
        fields = " ".join(
            f'c{j}: object(expression: "{oid}:{path}") {{ ... on Blob {{ text }} }}'
            for j, oid in enumerate(part)
        )
        q = ("query { rateLimit { cost remaining resetAt } "
             f"repository(owner: {json.dumps(owner)}, name: {json.dumps(name)}) {{ {fields} }} }}")
        repo = (gql(q).get("data") or {}).get("repository") or {}
        for j, oid in enumerate(part):
            obj = repo.get(f"c{j}")
            texts[oid] = obj.get("text") if obj else None
    return texts

dep_events, current_deps, manifest_commits, dev_repos, dev_profiles, errors = [], [], [], [], [], []

def process_repo(login, repo):
    owner, name = repo["nameWithOwner"].split("/")
    target = ((repo.get("defaultBranchRef") or {}).get("target")) or {}
    has = {}
    for alias, path in HIST_ALIAS.items():
        h = target.get(alias)
        has[path] = bool(h and h["nodes"])
        if not has[path]:
            continue
        eco, parser = PARSERS[path]
        commits = list(reversed(h["nodes"]))
        complete = not h["pageInfo"]["hasNextPage"]
        texts = fetch_texts(owner, name, path, [c["oid"] for c in commits])

        prev = None
        for c in commits:
            author = ((c.get("author") or {}).get("user") or {}).get("login")
            by_dev = bool(author) and author.lower() == login.lower()
            manifest_commits.append({
                "developer": login, "repo": repo["nameWithOwner"], "file": path, "oid": c["oid"],
                "committed_at": c["committedDate"], "author_login": author, "authored_by_dev": by_dev,
            })
            txt = texts.get(c["oid"])
            if txt is None:
                continue
            deps = parser(txt)
            base = {"developer": login, "repo": repo["nameWithOwner"], "file": path, "ecosystem": eco,
                    "date": c["committedDate"], "commit_oid": c["oid"], "authored_by_dev": by_dev,
                    "history_complete": complete}
            if prev is None:
                if complete:
                    for pkg, kind in deps.items():
                        dep_events.append({**base, "package": pkg, "package_id": f"{eco}:{pkg}",
                                           "dep_kind": kind, "action": "add", "is_initial_commit": True})
            else:
                for pkg in deps.keys() - prev.keys():
                    dep_events.append({**base, "package": pkg, "package_id": f"{eco}:{pkg}",
                                       "dep_kind": deps[pkg], "action": "add", "is_initial_commit": False})
                for pkg in prev.keys() - deps.keys():
                    dep_events.append({**base, "package": pkg, "package_id": f"{eco}:{pkg}",
                                       "dep_kind": prev[pkg], "action": "remove", "is_initial_commit": False})
            prev = deps

        newest_txt = texts.get(commits[-1]["oid"])
        if newest_txt is not None and prev is not None:
            for pkg, kind in prev.items():
                current_deps.append({"developer": login, "repo": repo["nameWithOwner"], "file": path,
                                     "ecosystem": eco, "package": pkg, "package_id": f"{eco}:{pkg}",
                                     "dep_kind": kind})

    dev_repos.append({
        "developer": login, "repo": repo["nameWithOwner"], "created_at": repo["createdAt"],
        "pushed_at": repo["pushedAt"], "stars": repo["stargazerCount"], "archived": repo["isArchived"],
        "language": (repo.get("primaryLanguage") or {}).get("name"),
        "has_package_json": has["package.json"], "has_requirements_txt": has["requirements.txt"],
        "has_pyproject_toml": has["pyproject.toml"],
    })

t0, cost0 = time.time(), api_stats["cost"]
for k, login in enumerate(tqdm(sample, desc="Développeurs"), 1):
    try:
        data = gql(REPOS_Q, {"login": login, "n": REPOS_PER_DEV, "h": HIST_PER_FILE}).get("data") or {}
        user = data.get("user")
        if not user:
            errors.append((login, "introuvable"))
            continue
        dev_profiles.append({"developer": login, "account_created_at": user["createdAt"],
                             "followers": user["followers"]["totalCount"],
                             "n_public_repos_owned": user["repositories"]["totalCount"]})
        for repo in user["repositories"]["nodes"]:
            process_repo(login, repo)
    except Exception as e:
        errors.append((login, repr(e)))
    if k % 25 == 0 and api_stats["cost"] > cost0:
        per_dev = (api_stats["cost"] - cost0) / k
        left = (len(sample) - k) * per_dev
        print(f"  [{k}/{len(sample)}] {per_dev:.1f} points/dév. | reste ≈ {left:,.0f} points (≈ {left/5000:.1f} h de quota)")

print(f"dév. traités : {len(dev_profiles)} | repos : {len(dev_repos)} | événements : {len(dep_events)} | erreurs : {len(errors)}")
print("API :", api_stats)

Développeurs:   0%|          | 0/580 [00:00<?, ?it/s]

  [25/580] 11.4 points/dév. | reste ≈ 6,349 points (≈ 1.3 h de quota)
  [50/580] 11.2 points/dév. | reste ≈ 5,947 points (≈ 1.2 h de quota)
  [75/580] 10.9 points/dév. | reste ≈ 5,488 points (≈ 1.1 h de quota)
  [100/580] 11.0 points/dév. | reste ≈ 5,285 points (≈ 1.1 h de quota)
  [125/580] 11.6 points/dév. | reste ≈ 5,296 points (≈ 1.1 h de quota)
  [150/580] 11.3 points/dév. | reste ≈ 4,868 points (≈ 1.0 h de quota)
  [175/580] 11.1 points/dév. | reste ≈ 4,480 points (≈ 0.9 h de quota)
  [200/580] 10.7 points/dév. | reste ≈ 4,079 points (≈ 0.8 h de quota)
  [225/580] 10.7 points/dév. | reste ≈ 3,785 points (≈ 0.8 h de quota)
  [250/580] 10.5 points/dév. | reste ≈ 3,473 points (≈ 0.7 h de quota)
  [275/580] 10.5 points/dév. | reste ≈ 3,213 points (≈ 0.6 h de quota)
  [300/580] 10.4 points/dév. | reste ≈ 2,914 points (≈ 0.6 h de quota)
  [325/580] 10.5 points/dév. | reste ≈ 2,676 points (≈ 0.5 h de quota)
  [350/580] 10.7 points/dév. | reste ≈ 2,457 points (≈ 0.5 h de quota)
  [375/58

## 8. Secondary signal: dated stars and forks

In [10]:
STARS_Q = """
query($login: String!, $after: String) {
  rateLimit { cost remaining resetAt }
  user(login: $login) {
    starredRepositories(first: 100, after: $after, orderBy: {field: STARRED_AT, direction: DESC}) {
      totalCount
      pageInfo { hasNextPage endCursor }
      edges { starredAt node {
        nameWithOwner createdAt stargazerCount description
        primaryLanguage { name }
        repositoryTopics(first: 10) { nodes { topic { name } } }
      } }
    }
  }
}
"""

FORKS_Q = """
query($login: String!) {
  rateLimit { cost remaining resetAt }
  user(login: $login) {
    repositories(first: 50, isFork: true, ownerAffiliations: [OWNER], privacy: PUBLIC,
                 orderBy: {field: CREATED_AT, direction: DESC}) {
      nodes { nameWithOwner createdAt parent { nameWithOwner } }
    }
  }
}
"""

stars, forks = [], []

for login in tqdm([d["developer"] for d in dev_profiles], desc="Stars / forks"):
    try:
        after = None
        for _ in range(STARS_PAGES):
            user = ((gql(STARS_Q, {"login": login, "after": after}).get("data") or {}).get("user")) or {}
            conn = user.get("starredRepositories")
            if not conn:
                break
            for e in conn["edges"]:
                n = e["node"]
                stars.append({
                    "developer": login, "repo": n["nameWithOwner"], "starred_at": e["starredAt"],
                    "repo_created_at": n["createdAt"], "repo_stars": n["stargazerCount"],
                    "language": (n.get("primaryLanguage") or {}).get("name"),
                    "topics": ",".join(t["topic"]["name"] for t in n["repositoryTopics"]["nodes"]),
                    "description": n.get("description"),
                })
            if not conn["pageInfo"]["hasNextPage"]:
                break
            after = conn["pageInfo"]["endCursor"]

        user = ((gql(FORKS_Q, {"login": login}).get("data") or {}).get("user")) or {}
        for n in (user.get("repositories") or {}).get("nodes", []):
            if n.get("parent"):
                forks.append({"developer": login, "repo": n["parent"]["nameWithOwner"],
                              "forked_at": n["createdAt"], "fork_name": n["nameWithOwner"]})
    except Exception as e:
        errors.append((login, "stars/forks : " + repr(e)))

print(f"stars : {len(stars)} | forks : {len(forks)} | erreurs cumulées : {len(errors)}")

Stars / forks:   0%|          | 0/580 [00:00<?, ?it/s]

stars : 26880 | forks : 6197 | erreurs cumulées : 0


## 9. AI tool catalogue: metadata for each encountered package
For **each distinct package** found in the dependencies, retrieve the following metadata:

### PyPI packages
- Classifiers
- Keywords
- Summary
- First release date
- Repository URL

### npm packages
- Keywords
- Description
- Repository URL
- First release date via `deps.dev` (if accessible)

### Performance requirements
- Use cached requests.
- Run 8 requests in parallel.

### `is_ai` classification rule

Use the following heuristic to determine whether a package is AI-related. The heuristic must be validated against `ai_label_sample.csv`.

A package is classified as AI-related (`is_ai = true`) if at least one of the following conditions is met:

1. **PyPI classifier:** It has the classifier:
   `Topic :: Scientific/Engineering :: Artificial Intelligence`

2. **Strong keyword:** Its summary or keywords contain at least one strong term, matching **whole words**:
   - `llm`
   - `openai`
   - `anthropic`
   - `langchain`
   - `pytorch`
   - `tensorflow`
   - `machine learning`
   - `deep learning`
   - `artificial intelligence`
   - `model context protocol`
   - `mcp`
   - Other equivalent AI-related terms.

3. **Weak keywords:** At least **2 distinct weak AI-related terms** are present.

4. **Seed list:** The package appears in the initial `AI_SEED` list.

**Important:** The `ai_reason` column must always explain why a package was classified as AI-related.

In [11]:
AI_SEED = {f"pypi:{p}" for p in """
torch transformers tensorflow keras jax scikit-learn langchain langchain-core langchain-community langgraph llama-index
openai anthropic google-generativeai google-genai huggingface-hub datasets accelerate peft trl vllm sentence-transformers
faiss-cpu chromadb pydantic-ai crewai autogen pyautogen ollama litellm mcp fastmcp diffusers onnxruntime xgboost lightgbm
bitsandbytes tiktoken gradio streamlit mlflow wandb ray instructor dspy openai-agents qdrant-client pinecone-client
haystack-ai guidance outlines semantic-kernel smolagents transformers-stream-generator unsloth langfuse langsmith
""".split()} | {f"npm:{p}" for p in """
openai @anthropic-ai/sdk langchain @langchain/core @langchain/openai @langchain/community @modelcontextprotocol/sdk ai
@ai-sdk/openai @ai-sdk/anthropic @huggingface/transformers @xenova/transformers ollama llamaindex @google/generative-ai
@mastra/core onnxruntime-node @tensorflow/tfjs @pinecone-database/pinecone chromadb @qdrant/js-client-rest
""".split()}

STRONG = r"(llms?|large language models?|openai|anthropic|claude|langchain|llama|pytorch|tensorflow|machine learning|deep learning|artificial intelligence|generative ai|genai|model context protocol|mcp|rag|retrieval[- ]augmented|ai agents?|agentic|huggingface|hugging face|neural networks?|diffusion models?)"
WEAK = r"(embeddings?|vector (?:database|store|search)|nlp|natural language processing|computer vision|reinforcement learning|fine[- ]tuning|gpt|gemini|transformers?|chatbot|speech recognition|text[- ]to[- ]speech|ai)"
STRONG_RE = re.compile(rf"(?<![\w-]){STRONG}(?![\w-])", re.I)
WEAK_RE = re.compile(rf"(?<![\w-]){WEAK}(?![\w-])", re.I)
AI_CLASSIFIER = "Topic :: Scientific/Engineering :: Artificial Intelligence"

def classify_ai(pkg_id, classifiers, keywords, text):
    reasons = []
    if pkg_id in AI_SEED:
        reasons.append("seed")
    if AI_CLASSIFIER in (classifiers or []):
        reasons.append("pypi_classifier")
    blob = f"{keywords or ''} {text or ''}"
    strong = sorted({m.group(0).lower() for m in STRONG_RE.finditer(blob)})
    weak = sorted({m.group(0).lower() for m in WEAK_RE.finditer(blob)} - set(strong))
    if strong:
        reasons.append("strong:" + "|".join(strong[:4]))
    elif len(weak) >= 2:
        reasons.append("weak:" + "|".join(weak[:4]))
    return bool(reasons), ";".join(reasons)

def _get_json(url, tries=3):
    for a in range(tries):
        try:
            r = SESS_META.get(url, timeout=30)
            if r.status_code == 404:
                return None
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** a * 2)
        except requests.RequestException:
            time.sleep(2 ** a)
    return None

SESS_META = requests.Session()

def fetch_meta(pkg_id):
    cpath = META_DIR / (hashlib.sha1(pkg_id.encode()).hexdigest() + ".json")
    if cpath.exists():
        return json.loads(cpath.read_text())
    eco, name = pkg_id.split(":", 1)
    m = {"package_id": pkg_id, "ecosystem": eco, "package": name, "found": False, "summary": None,
         "keywords": None, "classifiers": [], "first_release": None, "repo_url": None}
    if eco == "pypi":
        j = _get_json(f"https://pypi.org/pypi/{urllib.parse.quote(name)}/json")
        if j:
            info = j.get("info") or {}
            ups = [f.get("upload_time_iso_8601") for fl in (j.get("releases") or {}).values() for f in fl]
            urls = info.get("project_urls") or {}
            repo = next((v for k, v in urls.items() if v and "github.com" in v), None) or info.get("home_page")
            m.update(found=True, summary=info.get("summary"), keywords=info.get("keywords"),
                     classifiers=info.get("classifiers") or [], repo_url=repo,
                     first_release=min([u for u in ups if u], default=None))
    else:
        j = _get_json("https://registry.npmjs.org/" + urllib.parse.quote(name, safe="@").replace("/", "%2F") + "/latest")
        if j:
            kw = j.get("keywords")
            repo = j.get("repository")
            repo = repo.get("url") if isinstance(repo, dict) else repo
            m.update(found=True, summary=j.get("description"),
                     keywords=" ".join(kw) if isinstance(kw, list) else kw, repo_url=repo or j.get("homepage"))
        dd = _get_json("https://api.deps.dev/v3/systems/npm/packages/" + urllib.parse.quote(name, safe=""))
        if dd:
            pubs = [v.get("publishedAt") for v in dd.get("versions", []) if v.get("publishedAt")]
            m["first_release"] = min(pubs, default=None)
    m["is_ai"], m["ai_reason"] = classify_ai(pkg_id, m["classifiers"], m["keywords"], m["summary"])
    cpath.write_text(json.dumps(m))
    return m

all_pkgs = sorted({e["package_id"] for e in dep_events} | {c["package_id"] for c in current_deps})
print(f"{len(all_pkgs):,} packages distincts à documenter")
with ThreadPoolExecutor(8) as ex:
    metas = list(tqdm(ex.map(fetch_meta, all_pkgs), total=len(all_pkgs), desc="Métadonnées packages"))

df_pkgs = pd.DataFrame(metas)
df_pkgs["classifiers"] = df_pkgs["classifiers"].map(lambda c: "|".join(c or []))
print(f"trouvés : {df_pkgs.found.sum():,}/{len(df_pkgs):,} | marqués IA : {df_pkgs.is_ai.sum():,}")

10,914 packages distincts à documenter


Métadonnées packages:   0%|          | 0/10914 [00:00<?, ?it/s]

trouvés : 10,450/10,914 | marqués IA : 501


## 10. Export

In [12]:
df_events = pd.DataFrame(dep_events)
df_current = pd.DataFrame(current_deps)

if len(df_events):
    add_non_init = df_events[(df_events.action == "add") & (~df_events.is_initial_commit)]
    n_added = add_non_init.groupby(["commit_oid", "file"]).size().rename("commit_n_adds")
    df_events = df_events.merge(n_added, left_on=["commit_oid", "file"], right_index=True, how="left")
    df_events["commit_n_adds"] = df_events["commit_n_adds"].fillna(0).astype(int)
    df_events["is_bulk"] = df_events["commit_n_adds"] > BULK_THRESHOLD

pk = df_pkgs[["package_id", "is_ai", "ai_reason", "first_release"]]
for name in ("df_events", "df_current"):
    df = globals()[name]
    if len(df):
        globals()[name] = df.merge(pk, on="package_id", how="left")

dev_table = pd.DataFrame(dev_profiles)
if len(pre) and len(dev_table):
    dev_table = dev_table.merge(pre[["developer", "n_repos_with_manifest", "n_manifest_commits",
                                     "n_npm_commits", "n_pypi_commits"]], on="developer", how="left")

label_sample = df_pkgs[df_pkgs.found].sample(min(200, int(df_pkgs.found.sum())), random_state=SEED)[
    ["package_id", "summary", "keywords", "classifiers", "is_ai", "ai_reason"]].assign(label_manuel="")

outputs = {
    "developers.csv": dev_table,
    "dev_repos.csv": pd.DataFrame(dev_repos),
    "manifest_commits.csv": pd.DataFrame(manifest_commits),
    "dep_events.csv": df_events,
    "current_deps.csv": df_current,
    "packages.csv": df_pkgs,
    "ai_label_sample.csv": label_sample,
    "stars.csv": pd.DataFrame(stars),
    "forks.csv": pd.DataFrame(forks),
    "errors.csv": pd.DataFrame(errors, columns=["developer", "error"]),
}
for name, df in outputs.items():
    df.to_csv(OUT_DIR / name, index=False)
    print(f"{name:22s} {len(df):>9,} lignes")

# Contrôle qualité (taille des données, pas d'analyse)
if len(df_events):
    ok = df_events[(df_events.action == "add") & (~df_events.is_initial_commit) & df_events.authored_by_dev
                   & (df_events.dep_kind != "dev") & (~df_events.is_bulk)]
    per_dev = ok.groupby("developer").size()
    per_dev_ai = ok[ok.is_ai.fillna(False)].groupby("developer").size()
    print(f"\n[QC] ajouts propres : {len(ok):,} | développeurs : {ok.developer.nunique():,} | médiane/dév. : {per_dev.median():.0f}"
          f" | dév. ≥ 20 ajouts : {(per_dev >= 20).sum():,}")
    print(f"[QC] ajouts IA propres : {len(per_dev_ai) and int(per_dev_ai.sum()):,} | dév. avec ≥ 1 ajout IA : {len(per_dev_ai):,}"
          f" | dév. ≥ 5 ajouts IA : {(per_dev_ai >= 5).sum():,}")
print("\nAppels API réels :", api_stats["calls"], "| cache :", api_stats["cached"],
      "| coût cumulé :", api_stats["cost"], "| quota restant :", api_stats["remaining"])

developers.csv               580 lignes
dev_repos.csv              9,129 lignes
manifest_commits.csv      44,050 lignes
dep_events.csv           104,553 lignes
current_deps.csv          78,021 lignes
packages.csv              10,914 lignes
ai_label_sample.csv          200 lignes
stars.csv                 26,880 lignes
forks.csv                  6,197 lignes
errors.csv                     0 lignes

[QC] ajouts propres : 10,115 | développeurs : 493 | médiane/dév. : 13 | dév. ≥ 20 ajouts : 174
[QC] ajouts IA propres : 670 | dév. avec ≥ 1 ajout IA : 181 | dév. ≥ 5 ajouts IA : 38

Appels API réels : 7990 | cache : 44 | coût cumulé : 9862 | quota restant : 3772


## File dictionary (`out/`)

| File | Contents | Key columns |
|---|---|---|
| `prefilter.csv` | Accounts tested by the prefilter | developer, n_manifest_commits, n_repos_with_manifest |
| `developers.csv` | Selected developers | developer, followers, account_created_at, prefilter counters |
| `dev_repos.csv` | Personal, non-fork repositories | developer, repo, created_at, pushed_at, stars, language |
| `manifest_commits.csv` | Commits affecting a manifest file | developer, repo, file, committed_at, author_login, authored_by_dev |
| **`dep_events.csv`** | **Main interaction dataset** | developer, repo, package_id, dep_kind, **action**, **date**, is_initial_commit, authored_by_dev, **is_bulk**, commit_n_adds, **is_ai**, ai_reason, first_release |
| `current_deps.csv` | Current dependencies | developer, repo, package_id, dep_kind, is_ai |
| **`packages.csv`** | **Catalog of all packages encountered** | package_id, summary, keywords, classifiers, first_release, repo_url, is_ai, ai_reason |
| `ai_label_sample.csv` | 200 packages to be manually labeled (`label_manuel` column) | package_id, summary, is_ai, ai_reason |
| `stars.csv` / `forks.csv` | Secondary signals | developer, repo, starred_at / forked_at |
| `errors.csv` | Failures | developer, error |

### Recommended "genuine addition" filter for analysis

```python
(
    (df["action"] == "add") &
    (df["is_initial_commit"] == False) &
    (df["authored_by_dev"] == True) &
    (df["dep_kind"] != "dev") &
    (df["is_bulk"] == False)
)